# PROC RISK-style Python Notebook

This notebook is a **Python equivalent in structure** to a SAS `PROC RISK` workflow.

It models:

- market data setup
- account and instrument definitions
- trading rules
- scenario / path generation
- portfolio valuation
- risk measures

It is **not a literal SAS runtime clone**, but it is organized to feel like `PROC RISK`:
configuration → positions → methods → simulation → reports.


In [ ]:
# If needed:
# !pip install yfinance pandas numpy matplotlib

import math
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import yfinance as yf
except ImportError:
    raise ImportError("Install yfinance first: pip install yfinance")


In [ ]:
# -----------------------------
# PROC RISK-style configuration
# -----------------------------

TICKERS = ["BYND", "IBM", "PLAB", "AKAM", "TJX", "GRAB", "NTCO", "NUE", "RTX", "HWM", "CRS"]
START = "2022-01-01"
END = None
INITIAL_CAPITAL = 1_000_000
ROLLING_WINDOW = 63  # approx 3 months of trading days
WEIGHTS_MODE = "equal"  # or "custom"

CUSTOM_WEIGHTS = {
    "BYND": 0.05,
    "IBM": 0.10,
    "PLAB": 0.10,
    "AKAM": 0.10,
    "TJX": 0.10,
    "GRAB": 0.10,
    "NTCO": 0.10,
    "NUE": 0.10,
    "RTX": 0.10,
    "HWM": 0.10,
    "CRS": 0.05,
}


## Data layer

This corresponds to the market data / environment part of a SAS risk workflow.


In [ ]:
def download_prices(tickers: List[str], start: str, end: Optional[str] = None) -> pd.DataFrame:
    data = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    if len(tickers) == 1:
        close = data["Close"].to_frame(name=tickers[0])
    else:
        close = pd.concat({t: data[t]["Close"] for t in tickers if t in data.columns.get_level_values(0)}, axis=1)

    close = close.dropna(how="all").sort_index()
    close.columns.name = None
    return close

prices = download_prices(TICKERS, START, END)
prices.tail()


In [ ]:
returns = prices.pct_change().dropna()
smoothed_3m = prices.rolling(ROLLING_WINDOW).mean()

print("Prices shape:", prices.shape)
print("Returns shape:", returns.shape)


## Position / account definitions

This mimics the account and instrument definition layer.


In [ ]:
def make_weights(columns: List[str], mode: str = "equal", custom: Optional[Dict[str, float]] = None) -> pd.Series:
    if mode == "equal":
        w = pd.Series(1 / len(columns), index=columns, dtype=float)
    elif mode == "custom":
        if custom is None:
            raise ValueError("Custom weights requested but no CUSTOM_WEIGHTS provided.")
        w = pd.Series(custom, dtype=float).reindex(columns).fillna(0.0)
        total = w.sum()
        if total <= 0:
            raise ValueError("Custom weights sum must be positive.")
        w = w / total
    else:
        raise ValueError("mode must be 'equal' or 'custom'")
    return w

weights = make_weights(list(prices.columns), WEIGHTS_MODE, CUSTOM_WEIGHTS)
weights


## PROC RISK-like methods

Three account methods are implemented:

1. **Buy and hold**
2. **Convex account**: allocate more to assets above their 3-month smoothed average
3. **Concave account**: allocate more to assets below their 3-month smoothed average

These are practical Python analogues to formula-based account definitions.


In [ ]:
def normalize_positive(s: pd.Series) -> pd.Series:
    s = s.clip(lower=0)
    total = s.sum()
    if total == 0:
        return pd.Series(1 / len(s), index=s.index)
    return s / total

def convex_weights(price_row: pd.Series, smooth_row: pd.Series, base_weights: pd.Series) -> pd.Series:
    # Higher weight when price / smooth > 1
    ratio = (price_row / smooth_row).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    score = base_weights * ratio
    return normalize_positive(score)

def concave_weights(price_row: pd.Series, smooth_row: pd.Series, base_weights: pd.Series) -> pd.Series:
    # Higher weight when price / smooth < 1
    ratio = (smooth_row / price_row).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    score = base_weights * ratio
    return normalize_positive(score)

def backtest_dynamic_strategy(
    prices: pd.DataFrame,
    smooth: pd.DataFrame,
    base_weights: pd.Series,
    method: str,
    initial_capital: float = 1_000_000
):
    rets = prices.pct_change().fillna(0.0)
    value = pd.Series(index=prices.index, dtype=float)
    hist_weights = pd.DataFrame(index=prices.index, columns=prices.columns, dtype=float)

    value.iloc[0] = initial_capital
    hist_weights.iloc[0] = base_weights

    for i in range(1, len(prices)):
        date = prices.index[i]
        prev_date = prices.index[i - 1]

        prev_value = value.loc[prev_date]
        price_row = prices.loc[prev_date]
        smooth_row = smooth.loc[prev_date]

        if smooth_row.isna().all():
            w = base_weights
        else:
            if method == "buy_hold":
                w = base_weights
            elif method == "convex":
                w = convex_weights(price_row, smooth_row, base_weights)
            elif method == "concave":
                w = concave_weights(price_row, smooth_row, base_weights)
            else:
                raise ValueError("Unknown method")

        port_ret = float((w * rets.loc[date].fillna(0.0)).sum())
        value.loc[date] = prev_value * (1 + port_ret)
        hist_weights.loc[date] = w

    hist_weights.iloc[0] = base_weights
    return value, hist_weights

buyhold_value, buyhold_w = backtest_dynamic_strategy(prices, smoothed_3m, weights, "buy_hold", INITIAL_CAPITAL)
convex_value, convex_w = backtest_dynamic_strategy(prices, smoothed_3m, weights, "convex", INITIAL_CAPITAL)
concave_value, concave_w = backtest_dynamic_strategy(prices, smoothed_3m, weights, "concave", INITIAL_CAPITAL)


## Risk report layer

This is the reporting piece, analogous to the output stage of a SAS risk run.


In [ ]:
def risk_report(nav: pd.Series, alpha: float = 0.05) -> pd.Series:
    r = nav.pct_change().dropna()
    years = max((nav.index[-1] - nav.index[0]).days / 365.25, 1/365.25)
    total_return = nav.iloc[-1] / nav.iloc[0] - 1
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / years) - 1
    vol = r.std() * np.sqrt(252)
    sharpe = np.nan if vol == 0 else (r.mean() * 252) / vol

    running_max = nav.cummax()
    drawdown = nav / running_max - 1
    max_dd = drawdown.min()

    var_95 = r.quantile(alpha)
    cvar_95 = r[r <= var_95].mean() if (r <= var_95).any() else np.nan

    return pd.Series({
        "start_value": nav.iloc[0],
        "end_value": nav.iloc[-1],
        "total_return": total_return,
        "CAGR": cagr,
        "annual_volatility": vol,
        "sharpe_like": sharpe,
        "max_drawdown": max_dd,
        "VaR_95_daily": var_95,
        "CVaR_95_daily": cvar_95,
    })

report = pd.DataFrame({
    "buy_hold": risk_report(buyhold_value),
    "convex": risk_report(convex_value),
    "concave": risk_report(concave_value),
}).T

report


In [ ]:
ax = buyhold_value.plot(label="Buy & Hold", figsize=(12,6))
convex_value.plot(ax=ax, label="Convex")
concave_value.plot(ax=ax, label="Concave")
ax.set_title("PROC RISK-style Account Value Comparison")
ax.set_ylabel("Portfolio Value")
ax.legend()
plt.show()


In [ ]:
latest_smoothed = smoothed_3m.iloc[-1].rename("smoothed_3m_avg")
latest_price = prices.iloc[-1].rename("latest_price")
summary = pd.concat([latest_price, latest_smoothed, weights.rename("base_weight")], axis=1)
summary["price_vs_smooth_ratio"] = summary["latest_price"] / summary["smoothed_3m_avg"]
summary.sort_index()


## Scenario engine

A simple Monte Carlo scenario generator from empirical daily return moments.
This is the closest notebook analogue to a basic simulation block in a risk system.


In [ ]:
def simulate_paths_from_history(
    returns: pd.DataFrame,
    weights: pd.Series,
    horizon_days: int = 63,
    n_sims: int = 2000,
    initial_value: float = 1_000_000,
    seed: int = 42,
) -> pd.Series:
    rng = np.random.default_rng(seed)
    mu = returns.mean().values
    cov = returns.cov().values
    w = weights.reindex(returns.columns).fillna(0.0).values

    terminal_values = []
    for _ in range(n_sims):
        sim_rets = rng.multivariate_normal(mu, cov, size=horizon_days)
        port_rets = sim_rets @ w
        terminal = initial_value * np.prod(1 + port_rets)
        terminal_values.append(terminal)

    return pd.Series(terminal_values, name="terminal_value")

terminal_values = simulate_paths_from_history(returns, weights, horizon_days=63, n_sims=5000, initial_value=INITIAL_CAPITAL)
terminal_values.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])


In [ ]:
plt.figure(figsize=(10,6))
plt.hist(terminal_values, bins=50)
plt.title("63-Day Monte Carlo Terminal Portfolio Value")
plt.xlabel("Terminal Value")
plt.ylabel("Frequency")
plt.show()

print("Simulated 5% terminal quantile:", terminal_values.quantile(0.05))
print("Simulated 1% terminal quantile:", terminal_values.quantile(0.01))


## Mapping from SAS `PROC RISK` concepts to this notebook

| SAS-like idea | Python notebook analogue |
|---|---|
| Market data environment | `prices`, `returns`, `smoothed_3m` |
| Accounts / positions | `weights`, per-ticker portfolio state |
| Methods / formulas | `convex_weights`, `concave_weights`, backtest functions |
| Valuation | NAV time series from the strategy engine |
| Reports | `report` table |
| Simulation | `simulate_paths_from_history` |

For a stronger one-to-one emulation, the next step would be to add:

- factor models
- yield curves / fixed income instruments
- stress scenarios by date
- account hierarchies
- formal instrument classes


In [ ]:
# Optional export
report.to_csv("proc_risk_equivalent_report.csv")
summary.to_csv("proc_risk_equivalent_latest_summary.csv")
print("Saved CSV outputs.")
